# 🧠 洄瀾合成法：三個 AI + 公開高分解法 → 一個「新的」方法

不是照抄任何一支公開 notebook，而是用團隊三個 AI（立霧/秀姑巒/木瓜溪）讀完公開高分解法後，**synthesize 出一個新組合**，再用官方 metric 驗證。三方對照：
`baseline`(0.04) vs `improved`(我們的 watershed+分裂, 0.34) vs **`novel`(合成法)**。

**合成法 = 公開 LB628 的 keep-Z 古典偵測 ＋ 木瓜溪的雙向互惠連線 ＋ 秀姑巒/木瓜溪收斂的物理對稱分裂閘**。新巧思在後兩者：
- **雙向互惠連線**：t→t+1 與 t+1→t 互為最佳才連 → 殺掉密集區的錯連。
- **物理對稱分裂閘**：母細胞分裂成兩顆，兩顆會往**相反方向**分開；只有位移向量夠「反向對稱」才認定分裂 → 殺掉「擦身而過」的假分裂（我們之前量到的弱點）。

> 🙏 致謝：偵測技巧改編自公開 `yusuketogashi/lb628-clean-room-no-gpu-baseline`；連線/分裂巧思來自團隊 AI。需 Internet On（裝官方 metric）。純 CPU、不需 GPU。

In [ ]:
import subprocess, sys
def pip(*a):
    print('pip', *a); subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a])
pip('git+https://github.com/royerlab/tracksdata')
pip('--no-deps', 'git+https://github.com/royerlab/kaggle-cell-tracking-competition')
import tracksdata as td, polars as pl
from tracking_cellmot.metrics import evaluate, per_sample_metrics, summarise, node_recall
print('metric import OK')

In [ ]:
import os
from collections import defaultdict
import numpy as np
import zarr
from scipy.ndimage import uniform_filter, label, distance_transform_edt, gaussian_filter
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
try:
    from skimage.filters import threshold_otsu
except Exception:
    threshold_otsu = None

TRAIN = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'
SCALE = (1.625, 0.40625, 0.40625); SCALE_A = np.array(SCALE)
DOWNSAMPLE, PERCENTILE = 4, 90
MAX_LINK_DISTANCE, DIV_DISTANCE, GAP_DISTANCE, WS_MIN_DISTANCE = 15.0, 8.0, 20.0, 2
XY_DS, SMOOTH_SIGMA, MIN_PEAK_DIST, NMS_RADIUS_UM, REFINE_RZ, REFINE_RYX = 4, 1.0, 2, 2.65, 2, 5
DIV_COS_MAX = -0.2          # 兩子位移向量夾角 cos 要 < 此值（夠反向）才算分裂
MAX_SAMPLES = 1

def scaled_pairwise(A, B):
    d = A[:, None, :] - B[None, :, :]; return np.sqrt(((d * SCALE_A) ** 2).sum(axis=2))
def open_image(p): return zarr.open(p, mode='r')['0']
def read_n_total(geff_path):
    try:
        a = dict(zarr.open(geff_path, mode='r').attrs)
        def walk(d):
            if isinstance(d, dict):
                for k, v in d.items():
                    if 'estimated_number_of_nodes' in str(k): return v
                    r = walk(v)
                    if r is not None: return r
            return None
        v = walk(a)
        if v is not None: return float(v)
    except Exception: pass
    return float('nan')
print('setup ready')

In [ ]:
# === 偵測：baseline / watershed（我們的）/ combined（LB628 keep-Z 古典）===
def detect_baseline(vol):
    ds = vol[::DOWNSAMPLE, ::DOWNSAMPLE, ::DOWNSAMPLE]
    sm = uniform_filter(ds.astype(np.float32), size=3)
    b = sm > np.percentile(sm, PERCENTILE); lab, n = label(b)
    return [np.argwhere(lab == i).mean(0) * DOWNSAMPLE for i in range(1, n + 1) if (lab == i).any()]

def detect_watershed(vol):
    ds = vol[::DOWNSAMPLE, ::DOWNSAMPLE, ::DOWNSAMPLE]
    sm = uniform_filter(ds.astype(np.float32), size=3)
    b = sm > np.percentile(sm, PERCENTILE)
    if not b.any(): return []
    dist = distance_transform_edt(b); pk = peak_local_max(dist, min_distance=WS_MIN_DISTANCE, labels=b)
    if len(pk) == 0: return []
    mk = np.zeros(dist.shape, np.int32); mk[tuple(pk.T)] = np.arange(1, len(pk) + 1)
    lab = watershed(-dist, mk, mask=b)
    return [np.argwhere(lab == i).mean(0) * DOWNSAMPLE for i in range(1, int(lab.max()) + 1) if (lab == i).any()]

def _block_mean_xy(vol, f=XY_DS):
    z, y, x = vol.shape; y2, x2 = (y // f) * f, (x // f) * f
    a = vol[:, :y2, :x2].astype(np.float32)
    return a.reshape(z, y2 // f, f, x2 // f, f).mean(axis=(2, 4))
def _normalize(ds):
    lo, hi = np.percentile(ds, 1.0), np.percentile(ds, 99.8)
    x = np.clip((ds - lo) / max(hi - lo, 1e-6), 0.0, 1.0)
    return np.clip(x - gaussian_filter(x, sigma=6.0), 0.0, None)
def _refine(vol, coord):
    zc, yc, xc = [int(round(v)) for v in coord]
    z0, z1 = max(0, zc-REFINE_RZ), min(vol.shape[0], zc+REFINE_RZ+1)
    y0, y1 = max(0, yc-REFINE_RYX), min(vol.shape[1], yc+REFINE_RYX+1)
    x0, x1 = max(0, xc-REFINE_RYX), min(vol.shape[2], xc+REFINE_RYX+1)
    patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float32)
    if patch.size == 0: return coord
    w = np.clip(patch - np.percentile(patch, 20.0), 0.0, None); tot = float(w.sum())
    if tot <= 1e-6: return coord
    zz, yy, xx = np.indices(patch.shape, dtype=np.float32)
    return np.array([z0+float((zz*w).sum()/tot), y0+float((yy*w).sum()/tot), x0+float((xx*w).sum()/tot)])
def _physical_nms(coords, scores, radius_um):
    if len(coords) == 0: return coords
    cs = coords * SCALE_A; order = np.argsort(-scores); keep = []
    for i in order:
        if all(np.sqrt(((cs[i]-cs[j])**2).sum()) >= radius_um for j in keep): keep.append(i)
    return coords[keep]
def detect_combined(vol):
    sm = gaussian_filter(vol.astype(np.float32), sigma=(0.5, SMOOTH_SIGMA, SMOOTH_SIGMA))
    ds = _block_mean_xy(sm); norm = _normalize(ds)
    if norm.max() <= 0: return []
    thr = np.percentile(norm, 92.0)
    if threshold_otsu is not None:
        try: thr = max(thr, float(threshold_otsu(norm)))
        except Exception: pass
    peaks = peak_local_max(norm, min_distance=MIN_PEAK_DIST, threshold_abs=thr, exclude_border=False)
    if len(peaks) == 0: return []
    o = peaks.astype(np.float64); o[:, 1] = o[:, 1]*XY_DS + (XY_DS-1)/2.0; o[:, 2] = o[:, 2]*XY_DS + (XY_DS-1)/2.0
    refined = np.array([_refine(vol, c) for c in o])
    sc = np.array([vol[min(int(round(c[0])), vol.shape[0]-1), min(int(round(c[1])), vol.shape[1]-1), min(int(round(c[2])), vol.shape[2]-1)] for c in refined], dtype=np.float64)
    return [c for c in _physical_nms(refined, sc, NMS_RADIUS_UM)]
print('detection ready')

In [ ]:
# === 連線：舊版（baseline/improved 用）vs 合成新版（雙向互惠 + 物理對稱分裂閘）===
def _detect_all(arr, n_t, detect_fn):
    nodes = {}; fids, fxyz = [], []; nid = 1
    for t in range(n_t):
        cents = detect_fn(np.asarray(arr[t])); ids, xyz = [], []
        for c in cents:
            nodes[nid] = (t, float(c[0]), float(c[1]), float(c[2])); ids.append(nid); xyz.append(c); nid += 1
        fids.append(ids); fxyz.append(np.array(xyz) if xyz else np.empty((0, 3)))
    return nodes, fids, fxyz

def run_pipeline(arr, n_t, detect_fn, improved):
    nodes, fids, fxyz = _detect_all(arr, n_t, detect_fn)
    edges = []; has_in, out_count = set(), defaultdict(int)
    for t in range(n_t - 1):
        pid, pc = fids[t], fxyz[t]; cid, cc = fids[t+1], fxyz[t+1]
        if len(pid) == 0 or len(cid) == 0: continue
        D = scaled_pairwise(pc, cc); rr, c2 = linear_sum_assignment(D); mp, mc = set(), set()
        for ri, ci in zip(rr, c2):
            if D[ri, ci] <= MAX_LINK_DISTANCE:
                edges.append((pid[ri], cid[ci])); mp.add(ri); mc.add(ci); has_in.add(cid[ci]); out_count[pid[ri]] += 1
        if improved:
            for ci in range(len(cid)):
                if ci in mc: continue
                dd = np.sqrt((((pc-cc[ci])*SCALE_A)**2).sum(1)); j = int(np.argmin(dd))
                if j in mp and dd[j] <= DIV_DISTANCE and out_count[pid[j]] < 2:
                    edges.append((pid[j], cid[ci])); has_in.add(cid[ci]); out_count[pid[j]] += 1
    if improved:
        has_out = set(out_count.keys())
        for t in range(n_t - 2):
            ends = [(i, n_) for i, n_ in enumerate(fids[t]) if n_ not in has_out]
            starts = [(j, n_) for j, n_ in enumerate(fids[t+2]) if n_ not in has_in]
            if not ends or not starts: continue
            ec = fxyz[t][[i for i, _ in ends]]; sc = fxyz[t+2][[j for j, _ in starts]]
            D = scaled_pairwise(ec, sc); rr, c2 = linear_sum_assignment(D)
            for ri, ci in zip(rr, c2):
                if D[ri, ci] <= GAP_DISTANCE:
                    edges.append((ends[ri][1], starts[ci][1])); has_out.add(ends[ri][1]); has_in.add(starts[ci][1])
    return nodes, edges

def run_pipeline_novel(arr, n_t, detect_fn):
    """雙向互惠連線 + 物理對稱分裂閘 + gap closing。"""
    nodes, fids, fxyz = _detect_all(arr, n_t, detect_fn)
    edges = []; has_in, out_count = set(), defaultdict(int)
    for t in range(n_t - 1):
        pid, pc = fids[t], fxyz[t]; cid, cc = fids[t+1], fxyz[t+1]
        if len(pid) == 0 or len(cid) == 0: continue
        D = scaled_pairwise(pc, cc)
        fwd = D.argmin(axis=1)          # 每個母 → 最近子
        bwd = D.argmin(axis=0)          # 每個子 → 最近母
        mp, mc, child_disp = set(), set(), {}
        for i in range(len(pid)):       # 雙向互惠：互為最佳才連
            j = int(fwd[i])
            if int(bwd[j]) == i and D[i, j] <= MAX_LINK_DISTANCE:
                edges.append((pid[i], cid[j])); mp.add(i); mc.add(j)
                has_in.add(cid[j]); out_count[pid[i]] += 1; child_disp[i] = cc[j] - pc[i]
        # 物理對稱分裂閘：沒配到的子，緊鄰已配對母、且與既有子「反向分開」才算分裂
        for j in range(len(cid)):
            if j in mc: continue
            dd = np.sqrt((((pc-cc[j])*SCALE_A)**2).sum(1)); i = int(np.argmin(dd))
            if i in mp and dd[i] <= DIV_DISTANCE and out_count[pid[i]] < 2:
                v_new = (cc[j]-pc[i]) * SCALE_A; v_old = child_disp[i] * SCALE_A
                cos = float(np.dot(v_new, v_old) / (np.linalg.norm(v_new)*np.linalg.norm(v_old) + 1e-9))
                if cos < DIV_COS_MAX:   # 夠反向 → 真分裂
                    edges.append((pid[i], cid[j])); has_in.add(cid[j]); out_count[pid[i]] += 1
    has_out = set(out_count.keys())
    for t in range(n_t - 2):            # gap closing
        ends = [(i, n_) for i, n_ in enumerate(fids[t]) if n_ not in has_out]
        starts = [(j, n_) for j, n_ in enumerate(fids[t+2]) if n_ not in has_in]
        if not ends or not starts: continue
        ec = fxyz[t][[i for i, _ in ends]]; sc = fxyz[t+2][[j for j, _ in starts]]
        D = scaled_pairwise(ec, sc); rr, c2 = linear_sum_assignment(D)
        for ri, ci in zip(rr, c2):
            if D[ri, ci] <= GAP_DISTANCE:
                edges.append((ends[ri][1], starts[ci][1])); has_out.add(ends[ri][1]); has_in.add(starts[ci][1])
    return nodes, edges

def build_graph(nodes, edges):
    items = list(nodes.items()); id2idx = {nid: i for i, (nid, _) in enumerate(items)}
    g = td.graph.InMemoryGraph()
    for key in ['z', 'y', 'x']: g.add_node_attr_key(key, pl.Float64, -999999.0)
    tids = g.bulk_add_nodes([{'t': int(t), 'z': float(z), 'y': float(y), 'x': float(x)} for (_, (t, z, y, x)) in items])
    g.add_edge_attr_key('edge_prob', pl.Float64, 0.0)
    ed = [{'source_id': tids[id2idx[s]], 'target_id': tids[id2idx[d]], 'edge_prob': 1.0} for s, d in edges if s in id2idx and d in id2idx]
    if ed: g.bulk_add_edges(ed)
    return g
print('tracking (old + novel) ready')

In [ ]:
samples = sorted(d[:-5] for d in os.listdir(TRAIN) if d.endswith('.geff'))[:MAX_SAMPLES]
print('樣本：', samples, '\n')
results = {'baseline': [], 'improved': [], 'detectonly': [], 'novel': []}
import time
for name in samples:
    arr = open_image(os.path.join(TRAIN, name + '.zarr')); n_t = arr.shape[0]
    gt_res = td.graph.IndexedRXGraph.from_geff(os.path.join(TRAIN, name + '.geff'))
    gt = gt_res[0] if isinstance(gt_res, tuple) else gt_res
    n_total = read_n_total(os.path.join(TRAIN, name + '.geff'))
    plans = [('baseline', lambda a, n: run_pipeline(a, n, detect_baseline, False)),
             ('improved', lambda a, n: run_pipeline(a, n, detect_watershed, True)),
             ('detectonly', lambda a, n: run_pipeline(a, n, detect_combined, True)),  # LB628偵測+舊追蹤(對照)
             ('novel',    lambda a, n: run_pipeline_novel(a, n, detect_combined))]
    for tag, fn in plans:
        try:
            t0 = time.time(); nodes, edges = fn(arr, n_t); g = build_graph(nodes, edges)
            er = evaluate(g, gt, scale=SCALE, max_distance=7.0); recall = node_recall(g, gt)
            row = per_sample_metrics(er, n_total, recall); results[tag].append(row)
            print(f'{name} [{tag:9}] {time.time()-t0:6.1f}s  nodes={len(nodes):6d}  adj_edge_j={row.get("adj_edge_jaccard"):.4f}  recall={row.get("node_recall"):.3f}')
        except Exception as e:
            import traceback; traceback.print_exc(); print(f'{name} [{tag}] FAILED: {e}')

print('\n=== 三方官方分數（1 樣本全幀）===')
for tag in ['baseline', 'improved', 'detectonly', 'novel']:
    if results[tag]:
        s = summarise(results[tag]); print(f'{tag:9}: score={s.get("score"):.4f}')